<div style="text-align: center;">

# Social Network Analysis (CS342) | Assignment 6

## **Updated PageRank Algorithm**

---

**Student Name:** *Naishadh Rana*

**Roll No:** U23CS014

---

</div>


**Part I - Extended SVNIT web graph**
- Extend last lab's SVNIT crawl to include one-hop neighbors
- Build and visualize the extended directed web graph
- Run updated PageRank (with teleportation)

**Part II - Compare with simple PageRank**
- Run both algorithms on Karate Club and extended SVNIT graph
- Find highest and lowest PageRank nodes
- Compare score differences and top node overlap


**Required Libraries:** `networkx`, `matplotlib`, `requests`, `beautifulsoup4`


In [ ]:
import time
from collections import deque, Counter
from urllib.parse import urljoin, urlparse, urldefrag

import requests
from bs4 import BeautifulSoup
import networkx as nx
import matplotlib.pyplot as plt

print("Imported")


## Part I - Extended SVNIT Web Graph + Updated PageRank

We extend last lab's crawl by going one more hop deeper.

- Node = webpage URL
- Edge = hyperlink from one page to another

Basic PageRank formula:

$$PR_{t+1}(v) = \sum_{u\rightarrow v}\frac{PR_t(u)}{out(u)}$$

Updated PageRank (teleportation):

$$PR_{t+1}(v) = (1-d)\cdot\frac{1}{N} + d\sum_{u\rightarrow v}\frac{PR_t(u)}{out(u)}$$

where:
- $d$ = damping factor (e.g., 0.85)
- $N$ = total number of nodes
- $u\rightarrow v$ = edges pointing to node $v$
- $out(u)$ = out-degree of node $u$
- t = iteration number

This teleportation term helps avoid spider traps and dead ends.


In [ ]:
# helper functions

def normalize_url(url, base="https://www.svnit.ac.in/"):
    if not url:
        return None
    url = urljoin(base, url)
    url, _ = urldefrag(url)
    return url


def is_same_domain(url, domain="svnit.ac.in"):
    try:
        return urlparse(url).netloc.lower().endswith(domain)
    except:
        return False


def is_html_like(url):
    skip = (
        ".pdf", ".jpg", ".jpeg", ".png", ".gif", ".svg",
        ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
        ".zip", ".rar", ".7z"
    )
    path = urlparse(url).path.lower()
    return not any(path.endswith(ext) for ext in skip)


def extract_links(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        link = normalize_url(a.get("href"), base=base_url)
        if link:
            links.append(link)
    return links


def crawl_svnit(seed_url="https://www.svnit.ac.in/", max_pages=120, max_depth=3, request_timeout=10, sleep_sec=0.35):
    g = nx.DiGraph()
    visited = set()
    q = deque([(seed_url, 0)])

    while q and len(visited) < max_pages:
        url, depth = q.popleft()
        if url in visited or depth > max_depth:
            continue

        g.add_node(url)

        try:
            resp = requests.get(url, timeout=request_timeout, headers={"User-Agent": "SNA-Lab6"})
            if resp.status_code != 200:
                visited.add(url)
                continue
            html = resp.text
        except Exception:
            visited.add(url)
            continue

        visited.add(url)
        links = extract_links(html, url)

        for link in links:
            if is_same_domain(link) and is_html_like(link):
                g.add_edge(url, link)
                if link not in visited and depth + 1 <= max_depth:
                    q.append((link, depth + 1))

        time.sleep(sleep_sec)

    return g


In [ ]:
# build extended SVNIT graph
G_ext = crawl_svnit(max_pages=80, max_depth=3)

n = G_ext.number_of_nodes()
m = G_ext.number_of_edges()

indeg = dict(G_ext.in_degree())
outdeg = dict(G_ext.out_degree())

avg_in = sum(indeg.values()) / n if n else 0
avg_out = sum(outdeg.values()) / n if n else 0

print(f"Extended graph nodes: {n}")
print(f"Extended graph edges: {m}")
print(f"Average in-degree: {avg_in:.3f}")
print(f"Average out-degree: {avg_out:.3f}")


In [ ]:
# visualize extended graph (sample if large)
plt.figure(figsize=(10, 7))

if n <= 220:
    H = G_ext
else:
    H = G_ext.subgraph(list(G_ext.nodes())[:220]).copy()

pos = nx.spring_layout(H, seed=42, k=0.6)
nx.draw_networkx_nodes(H, pos, node_size=70, node_color="#85C4FFD7", alpha=0.9)
nx.draw_networkx_edges(H, pos, width=0.4, alpha=0.35, arrows=True, arrowsize=5)
plt.title("Extended SVNIT Web Graph")
plt.axis("off")
plt.show()


In [ ]:
# in/out degree distributions (with mean and median lines)
in_counts = Counter(indeg.values())
out_counts = Counter(outdeg.values())
in_vals = list(indeg.values())
out_vals = list(outdeg.values())

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.bar(in_counts.keys(), in_counts.values(), color="#F28E2B", alpha=0.7)
plt.axvline(sum(in_vals)/len(in_vals), color="#4AA942FF", linestyle="--", linewidth=1, label=f"Mean: {sum(in_vals)/len(in_vals):.2f}")
plt.axvline(sorted(in_vals)[len(in_vals)//2], color="#2A6C6DEF", linestyle="--", linewidth=1, label=f"Median: {sorted(in_vals)[len(in_vals)//2]}")
plt.title("In-degree")
plt.xlabel("Degree")
plt.ylabel("Count")
plt.legend()

plt.subplot(1, 2, 2)
plt.bar(out_counts.keys(), out_counts.values(), color="#F28E2B", alpha=0.7)
plt.axvline(sum(out_vals)/len(out_vals), color="#4AA942FF", linestyle="--", linewidth=1, label=f"Mean: {sum(out_vals)/len(out_vals):.2f}")
plt.axvline(sorted(out_vals)[len(out_vals)//2], color="#2A6C6DEF", linestyle="--", linewidth=1, label=f"Median: {sorted(out_vals)[len(out_vals)//2]}")
plt.title("Out-degree")
plt.xlabel("Degree")
plt.ylabel("Count")
plt.legend()

plt.tight_layout()
plt.show()


## Part II - Updated PageRank + Comparison

Now applying both versions:
- **Simple PageRank** (same style as Lab 5)
- **Updated PageRank** using teleportation matrix

Then we compare results on:
1. Karate Club graph
2. Extended SVNIT graph


In [ ]:
# simple PageRank (Lab 5 style)
def prank_simple(G, d=0.85, max_iter=100, tol=1e-6):
    if G.number_of_nodes() == 0:
        return {}

    nodes = list(G.nodes())
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}

    pr = [1.0 / n] * n
    out = {node: G.out_degree(node) for node in nodes}

    for _ in range(max_iter):
        new_pr = [0.0] * n

        sink = sum(pr[idx[node]] for node in nodes if out[node] == 0)

        for u in nodes:
            if out[u] == 0:
                continue
            share = pr[idx[u]] / out[u]
            for v in G.successors(u):
                new_pr[idx[v]] += share

        for i in range(n):
            new_pr[i] = (1 - d) / n + d * (new_pr[i] + sink / n)

        diff = sum(abs(new_pr[i] - pr[i]) for i in range(n))
        pr = new_pr
        if diff < tol:
            break

    return {node: pr[idx[node]] for node in nodes}


# updated PageRank (explicit teleportation step)
def prank_updated(G, d=0.85, max_iter=200, tol=1e-10):
    if G.number_of_nodes() == 0:
        return {}

    nodes = list(G.nodes())
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}

    pr = [1.0 / n] * n
    out = {node: G.out_degree(node) for node in nodes}

    for _ in range(max_iter):
        new_pr = [(1 - d) / n] * n  # teleportation base

        for u in nodes:
            u_i = idx[u]
            if out[u] == 0:
                # dangling node: spread uniformly
                share = d * pr[u_i] / n
                for i in range(n):
                    new_pr[i] += share
            else:
                share = d * pr[u_i] / out[u]
                for v in G.successors(u):
                    new_pr[idx[v]] += share

        diff = sum(abs(new_pr[i] - pr[i]) for i in range(n))
        pr = new_pr
        if diff < tol:
            break

    return {node: pr[idx[node]] for node in nodes}


def compare_pr(pr1, pr2, name1="simple", name2="updated", k=10):
    common = list(set(pr1.keys()) & set(pr2.keys()))
    if not common:
        print("No common nodes to compare")
        return

    avg_abs = sum(abs(pr1[n] - pr2[n]) for n in common) / len(common)
    top1 = [x[0] for x in sorted(pr1.items(), key=lambda x: x[1], reverse=True)[:k]]
    top2 = [x[0] for x in sorted(pr2.items(), key=lambda x: x[1], reverse=True)[:k]]
    overlap = len(set(top1) & set(top2))

    print(f"Avg absolute score difference ({name1} vs {name2}): {avg_abs:.8f}")
    print(f"Top-{k} overlap: {overlap}/{k}")


In [ ]:
# run both PageRank versions on Karate Club
K = nx.karate_club_graph().to_directed()

pr_k_simple = prank_simple(K, d=0.85, max_iter=200, tol=1e-8)
pr_k_updated = prank_updated(K, d=0.85, max_iter=500, tol=1e-12)

best_s = max(pr_k_simple, key=pr_k_simple.get)
worst_s = min(pr_k_simple, key=pr_k_simple.get)
best_u = max(pr_k_updated, key=pr_k_updated.get)
worst_u = min(pr_k_updated, key=pr_k_updated.get)

print("Karate Club")
print("-" * 35)
print(f"Simple PR highest: node {best_s} ({pr_k_simple[best_s]:.6f})")
print(f"Simple PR lowest : node {worst_s} ({pr_k_simple[worst_s]:.6f})")
print(f"Updated highest  : node {best_u} ({pr_k_updated[best_u]:.6f})")
print(f"Updated lowest   : node {worst_u} ({pr_k_updated[worst_u]:.6f})")

compare_pr(pr_k_simple, pr_k_updated)

print("\nUpdated PageRank vector (sorted):")
for node, score in sorted(pr_k_updated.items(), key=lambda x: x[1], reverse=True):
    print(f"  {node:>2}: {score:.6f}")


In [ ]:
# visualize updated PageRank on Karate Club
plt.figure(figsize=(8, 6))

sizes = [3000 * pr_k_updated[node] for node in K.nodes()]
pos = nx.spring_layout(K, seed=42)

nx.draw_networkx_nodes(K, pos, node_size=sizes, node_color="#EDC948", alpha=0.9)
nx.draw_networkx_edges(K, pos, width=0.5, alpha=0.5, arrows=False)
nx.draw_networkx_labels(K, pos, font_size=8)

plt.title("Karate Club (Updated PageRank)")
plt.axis("off")
plt.show()


In [ ]:
# run both PageRank versions on extended SVNIT graph
pr_w_simple = prank_simple(G_ext, d=0.85, max_iter=300, tol=1e-9)
pr_w_updated = prank_updated(G_ext, d=0.85, max_iter=600, tol=1e-12)

if pr_w_updated:
    best_ws = max(pr_w_simple, key=pr_w_simple.get)
    worst_ws = min(pr_w_simple, key=pr_w_simple.get)
    best_wu = max(pr_w_updated, key=pr_w_updated.get)
    worst_wu = min(pr_w_updated, key=pr_w_updated.get)

    print("Extended SVNIT Web Graph")
    print("-" * 35)
    print(f"Simple PR highest: {best_ws}")
    print(f"Simple PR lowest : {worst_ws}")
    print(f"Updated highest  : {best_wu}")
    print(f"Updated lowest   : {worst_wu}")

    compare_pr(pr_w_simple, pr_w_updated)

    print("\nTop 10 (updated):")
    for url, score in sorted(pr_w_updated.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"{score:.6f}  {url}")
else:
    print("Extended graph is empty")


### Notes / Findings

- Updated PageRank uses teleportation term $(1-d)/N$, so it is less sensitive to spider traps.
- On Karate Club, top-ranked nodes should stay similar to Lab 5 but scores can shift slightly.
- On extended SVNIT graph, updated PageRank is usually more stable for dangling-page heavy structures.
- Compare the printed `Top-10 overlap` and average absolute difference to explain simple vs updated behavior.
